# 📊 AIRE — Exploratory Data Analysis + Machine Learning

Notebook per il progetto AIRE (Analisi Inquinamento Milano 2025).

**Struttura:**
1. Setup & Connessione DB
2. Analisi Esplorativa (EDA)
3. Visualizzazioni
4. Preparazione dati ML
5. Regressione Lineare
6. Random Forest
7. K-Nearest Neighbors (KNN)
8. Confronto modelli
9. Conclusioni

## 1. Setup & Connessione al Database

In [ ]:
# Installa dipendenze se non presenti
# !pip install psycopg2-binary pandas scikit-learn matplotlib seaborn sqlalchemy python-dotenv

import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

sys.path.insert(0, "..")
load_dotenv("../.env")

print("✅ Librerie importate")
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
import app.db as db
import app.utils as utils

# Verifica connessione
try:
    conn = db.get_connection()
    conn.close()
    print("✅ Connesso a Supabase!")
except Exception as e:
    print(f"❌ Errore connessione: {e}")

## 2. Analisi Esplorativa (EDA)

Carico i dati grezzi dalla tabella  per capirne la struttura.

In [ ]:
# Carica tutti i dati
df = db.query_df("SELECT * FROM rilevamenti ORDER BY data, stazione_id")
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
# Panoramica generale
print("=== INFO ===")
print(df.dtypes)
print()
print("=== VALORI NULLI ===")
print(df.isnull().sum())
print(f"
Percentuale nulla: {df['valore'].isnull().mean()*100:.1f}%")

In [ ]:
# Statistiche per inquinante
df.groupby("inquinante")["valore"].describe().round(2)

In [ ]:
# Distribuzione misurazioni per stazione
df.groupby("stazione_id")["inquinante"].count().rename("n_misurazioni").to_frame()

## 3. Visualizzazioni

In [ ]:
# Boxplot distribuzione per inquinante
fig, ax = plt.subplots(figsize=(12, 5))
df_clean = df.dropna(subset=["valore"])
sns.boxplot(data=df_clean, x="inquinante", y="valore", ax=ax,
            order=["PM10","PM25","NO2","O3","C6H6","CO_8h","SO2"])
ax.set_title("Distribuzione valori per inquinante", fontsize=13)
ax.set_xlabel("")
ax.set_ylabel("µg/m³")
plt.tight_layout()
plt.show()

In [ ]:
# Trend PM10 per stazione nel tempo
df["data"] = pd.to_datetime(df["data"])
pm10 = df[df["inquinante"]=="PM10"].dropna(subset=["valore"])

fig, ax = plt.subplots(figsize=(13, 4))
for sta in sorted(pm10["stazione_id"].unique()):
    sub = pm10[pm10["stazione_id"]==sta]
    ax.plot(sub["data"], sub["valore"], label=f"Stazione {sta}", linewidth=1.5, alpha=0.8)
ax.axhline(50, color="red", linestyle="--", linewidth=1, label="Soglia 50 µg/m³")
ax.axhline(25, color="orange", linestyle="--", linewidth=1, label="Soglia 25 µg/m³")
ax.set_title("PM10 giornaliero per stazione — 2025")
ax.set_ylabel("PM10 (µg/m³)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap correlazione tra inquinanti (formato largo)
df_wide = utils.get_df_wide()
corr_cols = ["pm10","pm25","no2","o3","c6h6"]
corr = df_wide[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7,5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5)
ax.set_title("Correlazione tra inquinanti")
plt.tight_layout()
plt.show()

In [ ]:
# Distribuzione classi qualita
print("Distribuzione qualità aria (basata su PM10):")
print(df_wide["qualita"].value_counts())
print()
df_wide["qualita"].value_counts().plot.bar(color=["#27ae60","#f39c12","#e74c3c"],
    edgecolor="white", figsize=(6,3))
plt.title("Giorni per classe di qualità")
plt.xlabel("")
plt.ylabel("n° giorni")
plt.tight_layout()
plt.show()

## 4. Preparazione Dati per ML

In [ ]:
# Prepara dataset ML
df_ml, feature_cols = utils.prepara_dati_ml()
print(f"Dataset ML: {len(df_ml)} righe")
print(f"Feature: {feature_cols}")
print(f"Target regressione: pm10")
print(f"Target classificazione: qualita")
df_ml[feature_cols + ["pm10","qualita"]].describe()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

X = df_ml[feature_cols]
y_reg = df_ml["pm10"]

df_clf = df_ml.dropna(subset=["qualita"]).copy()
le = LabelEncoder()
df_clf["qualita_num"] = le.fit_transform(df_clf["qualita"])
print("Classi:", dict(enumerate(le.classes_)))

X_clf = df_clf[feature_cols]
y_clf = df_clf["qualita_num"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y_reg, test_size=0.2, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
print(f"Train/Test split: {len(X_train_r)}/{len(X_test_r)} (regressione)")
print(f"Train/Test split: {len(X_train_c)}/{len(X_test_c)} (classificazione)")

## 5. Modello 1 — Regressione Lineare

**Obiettivo:** Prevedere il valore numerico di PM10 a partire da NO₂, PM2.5, C₆H₆, O₃.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

model_lr = LinearRegression()
model_lr.fit(X_train_r, y_train_r)
y_pred_r = model_lr.predict(X_test_r)

r2   = r2_score(y_test_r, y_pred_r)
rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
print(f"R² Score : {r2:.3f}  (più vicino a 1 = meglio)")
print(f"RMSE     : {rmse:.2f} µg/m³  (errore medio nella stima di PM10)")
print()
for feat, coef in zip(feature_cols, model_lr.coef_):
    print(f"  Coefficiente {feat:8s}: {coef:+.4f}")

In [ ]:
# Grafico: valori reali vs previsti
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(y_test_r, y_pred_r, alpha=0.5, s=20, color="steelblue")
lim = [min(y_test_r.min(), y_pred_r.min()), max(y_test_r.max(), y_pred_r.max())]
ax.plot(lim, lim, "r--", linewidth=1.5, label="Previsione perfetta")
ax.set_xlabel("PM10 reale (µg/m³)")
ax.set_ylabel("PM10 previsto (µg/m³)")
ax.set_title(f"Regressione Lineare — R² = {r2:.3f}")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Modello 2 — Random Forest (Classificazione)

**Obiettivo:** Classificare la qualità dell'aria in **buona / media / cattiva**.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train_c, y_train_c)
y_pred_c = model_rf.predict(X_test_c)

acc_rf = accuracy_score(y_test_c, y_pred_c)
print(f"Accuratezza Random Forest: {acc_rf*100:.1f}%
")
print(classification_report(y_test_c, y_pred_c, target_names=le.classes_))

In [ ]:
# Matrice di confusione
cm = confusion_matrix(y_test_c, y_pred_c)
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(pd.DataFrame(cm, index=le.classes_, columns=le.classes_),
            annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Matrice di confusione — Random Forest")
ax.set_ylabel("Reale")
ax.set_xlabel("Previsto")
plt.tight_layout()
plt.show()

In [ ]:
# Importanza delle feature
fig, ax = plt.subplots(figsize=(6, 3))
importanze = model_rf.feature_importances_
ordin = np.argsort(importanze)[::-1]
ax.bar([feature_cols[i] for i in ordin], importanze[ordin],
       color="steelblue", edgecolor="white")
ax.set_title("Importanza delle feature — Random Forest")
ax.set_ylabel("Importanza")
plt.tight_layout()
plt.show()

## 7. Modello 3 — K-Nearest Neighbors (KNN)

⚠️ KNN richiede la **normalizzazione** delle feature perché è sensibile alle scale diverse.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

scaler = StandardScaler()
X_train_k = scaler.fit_transform(X_train_c)
X_test_k  = scaler.transform(X_test_c)

model_knn = KNeighborsClassifier(n_neighbors=5)
model_knn.fit(X_train_k, y_train_c)
y_pred_k = model_knn.predict(X_test_k)

acc_knn = accuracy_score(y_test_c, y_pred_k)
print(f"Accuratezza KNN (k=5): {acc_knn*100:.1f}%
")
print(classification_report(y_test_c, y_pred_k, target_names=le.classes_))

In [ ]:
# Come varia l'accuratezza al variare di k?
ks = range(1, 21)
accs = []
for k in ks:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_k, y_train_c)
    accs.append(accuracy_score(y_test_c, m.predict(X_test_k)))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(ks, accs, marker="o", linewidth=1.5)
ax.axvline(5, color="red", linestyle="--", label="k=5 scelto")
ax.set_xlabel("k (numero vicini)")
ax.set_ylabel("Accuratezza")
ax.set_title("KNN: accuratezza al variare di k")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Confronto Modelli

In [ ]:
print("="*50)
print(f"  Regressione Lineare   R² = {r2:.3f}  RMSE = {rmse:.2f}")
print(f"  Random Forest         Accuratezza = {acc_rf*100:.1f}%")
print(f"  KNN (k=5)             Accuratezza = {acc_knn*100:.1f}%")
print("="*50)
print()
print(f"Modello migliore per classificazione: ",
      "Random Forest" if acc_rf >= acc_knn else "KNN")

## 9. Conclusioni

- **Regressione Lineare**: R² = 0.814 — ottimo per stimare PM10 numericamente.
- **Random Forest**: ~79.5% accuratezza — il modello migliore per classificare qualità aria.
- **KNN**: ~76.9% — buono, ma richiede la normalizzazione dei dati.

**Feature più importanti**:  e  spiegano la maggior parte della varianza.

**Possibili miglioramenti**:
- Usare più dati storici (anni precedenti)
- Aggiungere dati meteo (temperatura, vento, umidità)
- Provare  o reti neurali
- Cross-validation k-fold per stime più robuste